In [ ]:
import numpy as np
import pandas as pd
import sklearn
import torch
import scipy.stats

import math

from tqdm import tqdm

from scipy.special import expit, log_expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator

In [ ]:
# PARAMETERS

NUM_RESAMPLES = 300
NUM_TRIALS = 100

SEED = 120013

TRAIN_PROP = 0.5 # rounded up to make even-sized collections
NUM_COLLECTIONS = 10

FONT_SIZE = 20

SAVE = True
LOAD = False

rng = np.random.default_rng(seed=SEED)

# Breast Cancer

### Generate semi-synthetic data

In [ ]:
df_bc = pd.read_csv('./data/breastcancer_data.csv')
df_bc

In [ ]:
X_bc = df_bc.iloc[:,1:].to_numpy()
X_bc = np.concatenate([np.ones(shape=(X_bc.shape[0],1)),X_bc],axis=1)
y_bc = df_bc.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_bc = np.load('saved_data/w_true_bc_ACTIVE.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bc[:,1:], y_bc)
    w_true_bc = np.ones(shape=(X_bc.shape[1],1))
    w_true_bc[0] = LR.intercept_
    w_true_bc[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bc = X_bc @ w_true_bc
probs_true_bc = expit(logit_true_bc)
om_probs_true_bc = expit(-logit_true_bc)

H_true_bc = - log_expit(logit_true_bc)  * probs_true_bc \
            - log_expit(-logit_true_bc) * om_probs_true_bc

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_bc = np.load('saved_data/logit_pred_bc_ACTIVE.npy')
    y_obs_bc      = np.load('saved_data/y_obs_bc_ACTIVE.npy')

else:
    logit_pred_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    y_obs_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_bc[:,i] = rng.binomial(1, probs_true_bc[:,0])
        LR.fit(X_bc, y_obs_bc[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_bc[:,i] = (X_bc @ w).reshape(-1)

In [ ]:
probs_pred_bc = expit(logit_pred_bc)
om_probs_pred_bc = expit(-logit_pred_bc)

### Active learning

In [ ]:
if LOAD:
    pred_logit_uni_bc = np.load('saved_data/pred_logit_uni_bc_ACTIVE.npy')
    pred_logit_reg_bc = np.load('saved_data/pred_logit_reg_bc_ACTIVE.npy')
    pred_logit_trg_bc = np.load('saved_data/pred_logit_trg_bc_ACTIVE.npy')
    w_obs_bc          = np.load('saved_data/w_obs_bc_ACTIVE.npy')

else:
    pred_logit_uni_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_reg_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_trg_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))

    w_obs_bc = np.zeros((X_bc.shape[1], NUM_TRIALS))

    sample_size = math.floor(X_bc.shape[0]*(1-TRAIN_PROP) / NUM_COLLECTIONS)
    train_size = X_bc.shape[0] - sample_size * NUM_COLLECTIONS
    
    for i in tqdm(range(NUM_TRIALS)):
        
        train = np.zeros((X_bc.shape[0],), dtype=bool)
        train[rng.choice(X_bc.shape[0],size=(train_size,),replace=False)] = 1
        
        LR_init = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
    
        LR_init.fit(X_bc[train], y_obs_bc[train,i])
    
        w_obs_bc[:,i] = LR_init.coef_.reshape(-1)
        
        train_uni = train.copy()
        train_reg = train.copy()
        train_trg = train.copy()
    
        w_uni = w_obs_bc[:,i].copy()
        w_reg = w_obs_bc[:,i].copy()
        w_trg = w_obs_bc[:,i].copy()
    
        w_all_active_reg = np.zeros((X_bc.shape[1], NUM_RESAMPLES))
        w_all_active_trg = np.zeros((X_bc.shape[1], NUM_RESAMPLES))
        
        pred_logit_uni_bc[:,i,0] = (X_bc @ w_obs_bc[:,i])
        pred_logit_reg_bc[:,i,0] = (X_bc @ w_obs_bc[:,i])
        pred_logit_trg_bc[:,i,0] = (X_bc @ w_obs_bc[:,i])
    
        for j in range(1, NUM_COLLECTIONS+1):
            probs_resamp = expit(X_bc[train_reg,:] @ w_reg)
            probs_true = probs_true_bc[train_trg]
            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_resamp).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_bc[train_reg], y_resamp)
    
                w_all_active_reg[:, k] = LR_active.coef_.reshape(-1)

            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_true).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_bc[train_trg], y_resamp)
    
                w_all_active_trg[:, k] = LR_active.coef_.reshape(-1)
    
            stds_pred = expit(X_bc[~train_reg] @ w_all_active_reg).std(axis=1).squeeze()
            stds_true = expit(X_bc[~train_trg] @ w_all_active_trg).std(axis=1).squeeze()
    
            sample_inds = rng.choice(np.arange(X_bc.shape[0])[~train_uni], size=(sample_size,), replace=False)
            train_uni[sample_inds] = 1
    
            sample_inds = np.arange(X_bc.shape[0])[~train_reg][np.argsort(stds_pred)[-sample_size:]]
            train_reg[sample_inds] = 1

            sample_inds = np.arange(X_bc.shape[0])[~train_trg][np.argsort(stds_true)[-sample_size:]]
            train_trg[sample_inds] = 1
    
            LR_uni = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_uni.fit(X_bc[train_uni], y_obs_bc[train_uni,i])
            w_uni = LR_uni.coef_.reshape(-1, 1)
            pred_logit_uni_bc[:,i,j] = (X_bc @ w_uni).squeeze()
    
            LR_reg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_reg.fit(X_bc[train_reg], y_obs_bc[train_reg,i])
            w_reg = LR_reg.coef_.reshape(-1, 1)
            pred_logit_reg_bc[:,i,j] = (X_bc @ w_reg).squeeze()

            LR_trg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_trg.fit(X_bc[train_trg], y_obs_bc[train_trg,i])
            w_trg = LR_trg.coef_.reshape(-1, 1)
            pred_logit_trg_bc[:,i,j] = (X_bc @ w_trg).squeeze()
    
            w_all_active_reg = np.zeros((X_bc.shape[1], NUM_RESAMPLES))
            w_all_active_trg = np.zeros((X_bc.shape[1], NUM_RESAMPLES))

In [ ]:
pred_probs_uni_bc = expit(pred_logit_uni_bc)
pred_probs_reg_bc = expit(pred_logit_reg_bc)
pred_probs_trg_bc = expit(pred_logit_trg_bc)

In [ ]:
avg_kl_div_uni_bc = np.mean(-probs_true_bc.reshape(probs_true_bc.shape + (1,))    * log_expit(pred_logit_uni_bc)
                         - om_probs_true_bc.reshape(probs_true_bc.shape + (1,)) * log_expit(-pred_logit_uni_bc)
                         - H_true_bc.reshape(probs_true_bc.shape + (1,)), axis=0)
avg_kl_div_reg_bc = np.mean(-probs_true_bc.reshape(probs_true_bc.shape + (1,))    * log_expit(pred_logit_reg_bc)
                         - om_probs_true_bc.reshape(probs_true_bc.shape + (1,)) * log_expit(-pred_logit_reg_bc)
                         - H_true_bc.reshape(probs_true_bc.shape + (1,)), axis=0)
avg_kl_div_trg_bc = np.mean(-probs_true_bc.reshape(probs_true_bc.shape + (1,))    * log_expit(pred_logit_trg_bc)
                         - om_probs_true_bc.reshape(probs_true_bc.shape + (1,)) * log_expit(-pred_logit_trg_bc)
                         - H_true_bc.reshape(probs_true_bc.shape + (1,)), axis=0)

# Bank

### Generate semi-synthetic data

In [ ]:
df_bk = pd.read_csv('./data/bank_data.csv')
df_bk

In [ ]:
X_bk = df_bk.iloc[:,1:].to_numpy()
X_bk = np.concatenate([np.ones(shape=(X_bk.shape[0],1)),X_bk],axis=1)
y_bk = df_bk.iloc[:,0].to_numpy()

In [ ]:
_, R = np.linalg.qr(X_bk)
X_bk = X_bk[:,np.abs(np.diag(R)) > 1e-10]

In [ ]:
if LOAD:
    w_true_bk = np.load('saved_data/w_true_bk_ACTIVE.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bk[:,1:], y_bk)
    w_true_bk = np.ones(shape=(X_bk.shape[1],1))
    w_true_bk[0] = LR.intercept_
    w_true_bk[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bk = X_bk @ w_true_bk
probs_true_bk = expit(logit_true_bk)
om_probs_true_bk = expit(-logit_true_bk)

H_true_bk = - log_expit(logit_true_bk)  * probs_true_bk \
            - log_expit(-logit_true_bk) * om_probs_true_bk

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_bk = np.load('saved_data/logit_pred_bk_ACTIVE.npy')
    y_obs_bk      = np.load('saved_data/y_obs_bk_ACTIVE.npy')

else:
    logit_pred_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    y_obs_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_bk[:,i] = rng.binomial(1, probs_true_bk[:,0])
        LR.fit(X_bk, y_obs_bk[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_bk[:,i] = (X_bk @ w).reshape(-1)

In [ ]:
probs_pred_bk = expit(logit_pred_bk)
om_probs_pred_bk = expit(-logit_pred_bk)

### Active learning

In [ ]:
if LOAD:
    pred_logit_uni_bk = np.load('saved_data/pred_logit_uni_bk_ACTIVE.npy')
    pred_logit_reg_bk = np.load('saved_data/pred_logit_reg_bk_ACTIVE.npy')
    pred_logit_trg_bk = np.load('saved_data/pred_logit_trg_bk_ACTIVE.npy')
    w_obs_bk          = np.load('saved_data/w_obs_bk_ACTIVE.npy')

else:
    pred_logit_uni_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_reg_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_trg_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))

    w_obs_bk = np.zeros((X_bk.shape[1], NUM_TRIALS))

    sample_size = math.floor(X_bk.shape[0]*(1-TRAIN_PROP) / NUM_COLLECTIONS)
    train_size = X_bk.shape[0] - sample_size * NUM_COLLECTIONS
    
    for i in tqdm(range(NUM_TRIALS)):
        
        train = np.zeros((X_bk.shape[0],), dtype=bool)
        train[rng.choice(X_bk.shape[0],size=(train_size,),replace=False)] = 1
        
        LR_init = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
    
        LR_init.fit(X_bk[train], y_obs_bk[train,i])
    
        w_obs_bk[:,i] = LR_init.coef_.reshape(-1)
        
        train_uni = train.copy()
        train_reg = train.copy()
        train_trg = train.copy()
    
        w_uni = w_obs_bk[:,i].copy()
        w_reg = w_obs_bk[:,i].copy()
        w_trg = w_obs_bk[:,i].copy()
    
        w_all_active_reg = np.zeros((X_bk.shape[1], NUM_RESAMPLES))
        w_all_active_trg = np.zeros((X_bk.shape[1], NUM_RESAMPLES))
        
        pred_logit_uni_bk[:,i,0] = (X_bk @ w_obs_bk[:,i])
        pred_logit_reg_bk[:,i,0] = (X_bk @ w_obs_bk[:,i])
        pred_logit_trg_bk[:,i,0] = (X_bk @ w_obs_bk[:,i])
    
        for j in range(1, NUM_COLLECTIONS+1):
            probs_resamp = expit(X_bk[train_reg,:] @ w_reg)
            probs_true = probs_true_bk[train_trg]
            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_resamp).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_bk[train_reg], y_resamp)
    
                w_all_active_reg[:, k] = LR_active.coef_.reshape(-1)

            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_true).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_bk[train_trg], y_resamp)
    
                w_all_active_trg[:, k] = LR_active.coef_.reshape(-1)
    
            stds_pred = expit(X_bk[~train_reg] @ w_all_active_reg).std(axis=1).squeeze()
            stds_true = expit(X_bk[~train_trg] @ w_all_active_trg).std(axis=1).squeeze()
    
            sample_inds = rng.choice(np.arange(X_bk.shape[0])[~train_uni], size=(sample_size,), replace=False)
            train_uni[sample_inds] = 1
    
            sample_inds = np.arange(X_bk.shape[0])[~train_reg][np.argsort(stds_pred)[-sample_size:]]
            train_reg[sample_inds] = 1

            sample_inds = np.arange(X_bk.shape[0])[~train_trg][np.argsort(stds_true)[-sample_size:]]
            train_trg[sample_inds] = 1
    
            LR_uni = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_uni.fit(X_bk[train_uni], y_obs_bk[train_uni,i])
            w_uni = LR_uni.coef_.reshape(-1, 1)
            pred_logit_uni_bk[:,i,j] = (X_bk @ w_uni).squeeze()
    
            LR_reg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_reg.fit(X_bk[train_reg], y_obs_bk[train_reg,i])
            w_reg = LR_reg.coef_.reshape(-1, 1)
            pred_logit_reg_bk[:,i,j] = (X_bk @ w_reg).squeeze()

            LR_trg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_trg.fit(X_bk[train_trg], y_obs_bk[train_trg,i])
            w_trg = LR_trg.coef_.reshape(-1, 1)
            pred_logit_trg_bk[:,i,j] = (X_bk @ w_trg).squeeze()
    
            w_all_active_reg = np.zeros((X_bk.shape[1], NUM_RESAMPLES))
            w_all_active_trg = np.zeros((X_bk.shape[1], NUM_RESAMPLES))

In [ ]:
pred_probs_uni_bk = expit(pred_logit_uni_bk)
pred_probs_reg_bk = expit(pred_logit_reg_bk)
pred_probs_trg_bk = expit(pred_logit_trg_bk)

In [ ]:
avg_kl_div_uni_bk = np.mean(-probs_true_bk.reshape(probs_true_bk.shape + (1,))    * log_expit(pred_logit_uni_bk)
                         - om_probs_true_bk.reshape(probs_true_bk.shape + (1,)) * log_expit(-pred_logit_uni_bk)
                         - H_true_bk.reshape(probs_true_bk.shape + (1,)), axis=0)
avg_kl_div_reg_bk = np.mean(-probs_true_bk.reshape(probs_true_bk.shape + (1,))    * log_expit(pred_logit_reg_bk)
                         - om_probs_true_bk.reshape(probs_true_bk.shape + (1,)) * log_expit(-pred_logit_reg_bk)
                         - H_true_bk.reshape(probs_true_bk.shape + (1,)), axis=0)
avg_kl_div_trg_bk = np.mean(-probs_true_bk.reshape(probs_true_bk.shape + (1,))    * log_expit(pred_logit_trg_bk)
                         - om_probs_true_bk.reshape(probs_true_bk.shape + (1,)) * log_expit(-pred_logit_trg_bk)
                         - H_true_bk.reshape(probs_true_bk.shape + (1,)), axis=0)

# Apnea

### Generate semi-synthetic data

In [ ]:
df_ap = pd.read_csv('./data/apnea/apnea_data.csv')
df_ap

In [ ]:
df_ap = df_ap.drop(columns=['Age','Sex','bmi_lt_25']).join(pd.get_dummies(df_ap.loc[:,['Age','Sex']], drop_first=True, dtype=int))
df_ap

In [ ]:
X_ap = df_ap.iloc[:,1:].to_numpy()
X_ap = np.concatenate([np.ones(shape=(X_ap.shape[0],1)),X_ap],axis=1)
y_ap = df_ap.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_ap = np.load('saved_data/w_true_ap_ACTIVE.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_ap[:,1:], y_ap)
    w_true_ap = np.ones(shape=(X_ap.shape[1],1))
    w_true_ap[0] = LR.intercept_
    w_true_ap[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_ap = X_ap @ w_true_ap
probs_true_ap = expit(logit_true_ap)
om_probs_true_ap = expit(-logit_true_ap)

H_true_ap = - log_expit(logit_true_ap)  * probs_true_ap \
            - log_expit(-logit_true_ap) * om_probs_true_ap

### Initial logreg on semi-synthetic data

In [ ]:
if LOAD:
    logit_pred_ap = np.load('saved_data/logit_pred_ap_ACTIVE.npy')
    y_obs_ap      = np.load('saved_data/y_obs_ap_ACTIVE.npy')

else:
    logit_pred_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    y_obs_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        LR = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
        y_obs_ap[:,i] = rng.binomial(1, probs_true_ap[:,0])
        LR.fit(X_ap, y_obs_ap[:,i])
        w = LR.coef_.reshape(-1, 1)
    
        logit_pred_ap[:,i] = (X_ap @ w).reshape(-1)

In [ ]:
probs_pred_ap = expit(logit_pred_ap)
om_probs_pred_ap = expit(-logit_pred_ap)

### Active learning tests

In [ ]:
if LOAD:
    pred_logit_uni_ap = np.load('saved_data/pred_logit_uni_ap_ACTIVE.npy')
    pred_logit_reg_ap = np.load('saved_data/pred_logit_reg_ap_ACTIVE.npy')
    pred_logit_trg_ap = np.load('saved_data/pred_logit_trg_ap_ACTIVE.npy')
    w_obs_ap          = np.load('saved_data/w_obs_ap_ACTIVE.npy')

else:
    pred_logit_uni_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_reg_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))
    pred_logit_trg_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_COLLECTIONS+1))

    w_obs_ap = np.zeros((X_ap.shape[1], NUM_TRIALS))

    sample_size = math.floor(X_ap.shape[0]*(1-TRAIN_PROP) / NUM_COLLECTIONS)
    train_size = X_ap.shape[0] - sample_size * NUM_COLLECTIONS
    
    for i in tqdm(range(NUM_TRIALS)):
        
        train = np.zeros((X_ap.shape[0],), dtype=bool)
        train[rng.choice(X_ap.shape[0],size=(train_size,),replace=False)] = 1
        
        LR_init = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
    
        LR_init.fit(X_ap[train], y_obs_ap[train,i])
    
        w_obs_ap[:,i] = LR_init.coef_.reshape(-1)
        
        train_uni = train.copy()
        train_reg = train.copy()
        train_trg = train.copy()
    
        w_uni = w_obs_ap[:,i].copy()
        w_reg = w_obs_ap[:,i].copy()
        w_trg = w_obs_ap[:,i].copy()
    
        w_all_active_reg = np.zeros((X_ap.shape[1], NUM_RESAMPLES))
        w_all_active_trg = np.zeros((X_ap.shape[1], NUM_RESAMPLES))
        
        pred_logit_uni_ap[:,i,0] = (X_ap @ w_obs_ap[:,i])
        pred_logit_reg_ap[:,i,0] = (X_ap @ w_obs_ap[:,i])
        pred_logit_trg_ap[:,i,0] = (X_ap @ w_obs_ap[:,i])
    
        for j in range(1, NUM_COLLECTIONS+1):
            probs_resamp = expit(X_ap[train_reg,:] @ w_reg)
            probs_true = probs_true_ap[train_trg]
            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_resamp).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_ap[train_reg], y_resamp)
    
                w_all_active_reg[:, k] = LR_active.coef_.reshape(-1)

            for k in range(NUM_RESAMPLES):
                y_resamp = rng.binomial(1, probs_true).squeeze()
    
                LR_active = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
                LR_active.fit(X_ap[train_trg], y_resamp)
    
                w_all_active_trg[:, k] = LR_active.coef_.reshape(-1)
    
            stds_pred = expit(X_ap[~train_reg] @ w_all_active_reg).std(axis=1).squeeze()
            stds_true = expit(X_ap[~train_trg] @ w_all_active_trg).std(axis=1).squeeze()
    
            sample_inds = rng.choice(np.arange(X_ap.shape[0])[~train_uni], size=(sample_size,), replace=False)
            train_uni[sample_inds] = 1
    
            sample_inds = np.arange(X_ap.shape[0])[~train_reg][np.argsort(stds_pred)[-sample_size:]]
            train_reg[sample_inds] = 1

            sample_inds = np.arange(X_ap.shape[0])[~train_trg][np.argsort(stds_true)[-sample_size:]]
            train_trg[sample_inds] = 1
    
            LR_uni = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_uni.fit(X_ap[train_uni], y_obs_ap[train_uni,i])
            w_uni = LR_uni.coef_.reshape(-1, 1)
            pred_logit_uni_ap[:,i,j] = (X_ap @ w_uni).squeeze()
    
            LR_reg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_reg.fit(X_ap[train_reg], y_obs_ap[train_reg,i])
            w_reg = LR_reg.coef_.reshape(-1, 1)
            pred_logit_reg_ap[:,i,j] = (X_ap @ w_reg).squeeze()

            LR_trg = LogisticRegression(C=np.inf, fit_intercept=False, solver='lbfgs')
            LR_trg.fit(X_ap[train_trg], y_obs_ap[train_trg,i])
            w_trg = LR_trg.coef_.reshape(-1, 1)
            pred_logit_trg_ap[:,i,j] = (X_ap @ w_trg).squeeze()
    
            w_all_active_reg = np.zeros((X_ap.shape[1], NUM_RESAMPLES))
            w_all_active_trg = np.zeros((X_ap.shape[1], NUM_RESAMPLES))

In [ ]:
pred_probs_uni_ap = expit(pred_logit_uni_ap)
pred_probs_reg_ap = expit(pred_logit_reg_ap)
pred_probs_trg_ap = expit(pred_logit_trg_ap)

In [ ]:
avg_kl_div_uni_ap = np.mean(-probs_true_ap.reshape(probs_true_ap.shape + (1,))    * log_expit(pred_logit_uni_ap)
                         - om_probs_true_ap.reshape(probs_true_ap.shape + (1,)) * log_expit(-pred_logit_uni_ap)
                         - H_true_ap.reshape(probs_true_ap.shape + (1,)), axis=0)
avg_kl_div_reg_ap = np.mean(-probs_true_ap.reshape(probs_true_ap.shape + (1,))    * log_expit(pred_logit_reg_ap)
                         - om_probs_true_ap.reshape(probs_true_ap.shape + (1,)) * log_expit(-pred_logit_reg_ap)
                         - H_true_ap.reshape(probs_true_ap.shape + (1,)), axis=0)
avg_kl_div_trg_ap = np.mean(-probs_true_ap.reshape(probs_true_ap.shape + (1,))    * log_expit(pred_logit_trg_ap)
                         - om_probs_true_ap.reshape(probs_true_ap.shape + (1,)) * log_expit(-pred_logit_trg_ap)
                         - H_true_ap.reshape(probs_true_ap.shape + (1,)), axis=0)

### Save data

In [ ]:
if SAVE:
    np.save('saved_data/w_true_bc_ACTIVE.npy', w_true_bc)
    np.save('saved_data/logit_pred_bc_ACTIVE.npy', logit_pred_bc)
    np.save('saved_data/y_obs_bc_ACTIVE.npy', y_obs_bc)
    np.save('saved_data/pred_logit_uni_bc_ACTIVE.npy', pred_logit_uni_bc)
    np.save('saved_data/pred_logit_reg_bc_ACTIVE.npy', pred_logit_reg_bc)
    np.save('saved_data/pred_logit_trg_bc_ACTIVE.npy', pred_logit_trg_bc)
    np.save('saved_data/w_obs_bc_ACTIVE.npy', w_obs_bc)

    np.save('saved_data/w_true_bk_ACTIVE.npy', w_true_bk)
    np.save('saved_data/logit_pred_bk_ACTIVE.npy', logit_pred_bk)
    np.save('saved_data/y_obs_bk_ACTIVE.npy', y_obs_bk)
    np.save('saved_data/pred_logit_uni_bk_ACTIVE.npy', pred_logit_uni_bk)
    np.save('saved_data/pred_logit_reg_bk_ACTIVE.npy', pred_logit_reg_bk)
    np.save('saved_data/pred_logit_trg_bk_ACTIVE.npy', pred_logit_trg_bk)
    np.save('saved_data/w_obs_bk_ACTIVE.npy', w_obs_bk)

    np.save('saved_data/w_true_ap_ACTIVE.npy', w_true_ap)
    np.save('saved_data/logit_pred_ap_ACTIVE.npy', logit_pred_ap)
    np.save('saved_data/y_obs_ap_ACTIVE.npy', y_obs_ap)
    np.save('saved_data/pred_logit_uni_ap_ACTIVE.npy', pred_logit_uni_ap)
    np.save('saved_data/pred_logit_reg_ap_ACTIVE.npy', pred_logit_reg_ap)
    np.save('saved_data/pred_logit_trg_ap_ACTIVE.npy', pred_logit_trg_ap)
    np.save('saved_data/w_obs_ap_ACTIVE.npy', w_obs_ap)

# Plots

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_bc, axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_bc, axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_bc, axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_bc, 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_bc, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_bc, 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_bc, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_bc, 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_bc, 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Average KL-Divergence')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

plt.rcParams.update({'font.size': FONT_SIZE})

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_bk, axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_bk, axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_bk, axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_bk, 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_bk, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_bk, 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_bk, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_bk, 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_bk, 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Average KL-Divergence')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

plt.rcParams.update({'font.size': FONT_SIZE})

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_ap, axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_ap, axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_ap, axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_ap, 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_ap, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_ap, 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_ap, 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_ap, 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_ap, 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Average KL-Divergence')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_bc - avg_kl_div_trg_bc[:,-1].reshape(-1,1), axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_bc - avg_kl_div_reg_bc[:,-1].reshape(-1,1), axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_bc - avg_kl_div_uni_bc[:,-1].reshape(-1,1), axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_bc - avg_kl_div_trg_bc[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_bc - avg_kl_div_trg_bc[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_bc - avg_kl_div_reg_bc[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_bc - avg_kl_div_reg_bc[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_bc - avg_kl_div_uni_bc[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_bc - avg_kl_div_uni_bc[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Excess average KL-Div.')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig('plots/active-learning-bc.pdf', bbox_inches='tight')

plt.show()

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_bk - avg_kl_div_trg_bk[:,-1].reshape(-1,1), axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_bk - avg_kl_div_reg_bk[:,-1].reshape(-1,1), axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_bk - avg_kl_div_uni_bk[:,-1].reshape(-1,1), axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_bk - avg_kl_div_trg_bk[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_bk - avg_kl_div_trg_bk[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_bk - avg_kl_div_reg_bk[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_bk - avg_kl_div_reg_bk[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_bk - avg_kl_div_uni_bk[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_bk - avg_kl_div_uni_bk[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Excess average KL-Div.')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig('plots/active-learning-bk.pdf', bbox_inches='tight')

plt.show()

plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_trg_ap - avg_kl_div_trg_ap[:,-1].reshape(-1,1), axis=0),
         'o--',
         label='Using regret (true)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_reg_ap - avg_kl_div_reg_ap[:,-1].reshape(-1,1), axis=0),
         '*:',
         label='Using regret (est.)')
plt.plot(np.arange(NUM_COLLECTIONS + 1),
         np.median(avg_kl_div_uni_ap - avg_kl_div_uni_ap[:,-1].reshape(-1,1), axis=0),
         'x-',
         label='Uniform sampling',
         color='black')
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_trg_ap - avg_kl_div_trg_ap[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_trg_ap - avg_kl_div_trg_ap[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_reg_ap - avg_kl_div_reg_ap[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_reg_ap - avg_kl_div_reg_ap[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2)
plt.fill_between(np.arange(NUM_COLLECTIONS + 1),
                 np.quantile(avg_kl_div_uni_ap - avg_kl_div_uni_ap[:,-1].reshape(-1,1), 0.25, axis=0),
                 np.quantile(avg_kl_div_uni_ap - avg_kl_div_uni_ap[:,-1].reshape(-1,1), 0.75, axis=0),
                 alpha=0.2,
                 color='black')
plt.xlabel('Number of additional collections')
plt.ylabel('Excess average KL-Div.')
plt.legend()
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig('plots/active-learning-ap.pdf', bbox_inches='tight')

plt.show()